# EmojiMuse — Backend on Colab

Runs the FastAPI + MusicGen backend and exposes it with a public URL.

**Before you start:** `Runtime → Change runtime type → T4 GPU → Save`.

Then run the cells top to bottom.

## 1. Check you have a GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU! Runtime → Change runtime type → T4 GPU, then re-run."
print("GPU OK:", torch.cuda.get_device_name(0))

## 2. Get the backend code onto Colab

**Pick ONE option**, then run the cell below.

- **Option A — Upload a zip:** on your PC, zip the `backend` folder → `backend.zip`.
  Click the folder icon 📁 in the left sidebar → **Upload** → choose `backend.zip`.
- **Option B — GitHub:** edit `GITHUB_URL` below to your repo.

The cell auto-detects which one you used.

In [ ]:
import os, subprocess

GITHUB_URL = ""  # e.g. "https://github.com/yourname/EmojiMuse.git"  (leave blank if uploading a zip)

os.chdir("/content")

if os.path.isdir("/content/backend"):
    pass
elif os.path.isfile("/content/backend.zip"):
    subprocess.run(["unzip", "-q", "-o", "backend.zip"], check=True)
elif GITHUB_URL:
    subprocess.run(["git", "clone", "--depth", "1", GITHUB_URL, "repo"], check=True)
    # find the backend folder inside the cloned repo
    for root, dirs, files in os.walk("/content/repo"):
        if "colab_server.py" in files:
            os.symlink(root, "/content/backend")
            break
else:
    raise SystemExit("Upload backend.zip via the 📁 Files panel, or set GITHUB_URL, then re-run.")

os.chdir("/content/backend")
print("Working in:", os.getcwd())
print("Contents:", sorted(os.listdir()))

## 3. Install dependencies

Takes 2–4 minutes. torch is already on Colab, so this mostly installs FastAPI and the tunnel.

In [ ]:
!pip install -q -r requirements.txt

## 4. (Optional) tunnel choice

Default is **cloudflared** — no signup, nothing to configure. Skip this cell unless you
specifically want ngrok (then paste your token from dashboard.ngrok.com).

In [ ]:
import os
# os.environ["EMOJIMUSE_TUNNEL"] = "ngrok"
# os.environ["NGROK_AUTHTOKEN"] = "paste-your-token-here"

# Other knobs you can uncomment:
# os.environ["EMOJIMUSE_DEFAULT_DURATION"] = "10"   # clip length in seconds
# os.environ["EMOJIMUSE_MAX_DURATION"] = "30"
print("tunnel:", os.environ.get("EMOJIMUSE_TUNNEL", "cloudflared"))

## 5. Start the server

This cell **keeps running** — that's correct, it *is* the server.

Wait ~1–2 min (first run downloads the MusicGen model), then look for:

```
  EmojiMuse API is live at:  https://xxxx.trycloudflare.com
```

**Copy that URL.** To stop the server, press the ⏹ stop button on this cell.

In [ ]:
!python colab_server.py

## 6. Test it

1. Open `https://xxxx.trycloudflare.com/docs` in a new browser tab.
2. **POST /generate → Try it out**, body:
   ```json
   { "emoji": "🌧️💔🌙", "duration": 10 }
   ```
3. **Execute.** You get back a `prompt` and an `audio_url`.
4. Paste `https://xxxx.trycloudflare.com` + the `audio_url` into a browser tab to hear the track.

## 7. Connect the frontend

Open `Emoji.html`, go to the **Rig / settings** dialog, paste the
`https://xxxx.trycloudflare.com` URL, save.

The URL changes every time Colab reconnects — just re-paste the new one. That's normal.